## Search & Amplitude Amplification Algorithms
Algorithms in this branch boost the amplitude of target computational states, providing quadratic speedups for unstructured problems.

### 1. [Grover’s Search Algorithm](https://en.wikipedia.org/wiki/Grover%27s_algorithm)
* **Problem Domain:** Unstructured Database Search.
* **Core Function:** Finds a target element $w$ in an unsorted space of size $N$ in $\mathcal{O}(\sqrt{N})$ queries.
* **Algorithmic Mechanism:**
    1. Prepare equal superposition $\vert{}\psi\rangle = H^{\otimes n}\vert{}0\rangle$.
    2. Apply Oracle operator $U_w = I - 2\vert{}w\rangle\langle w\vert{}$ (flips target phase).
    3. Apply Diffusion operator $U_s = 2\vert{}\psi\rangle\langle\psi\vert{} - I$ (amplifies target amplitude).
    4. Repeat step 2-3 approximately $\frac{\pi}{4}\sqrt{N}$ times before measuring.

In [1]:
from qiskit import QuantumCircuit

# Search for state |11> in 2-qubit space
qc = QuantumCircuit(2, 2)

# Step 1: Equal Superposition
qc.h([0, 1])

# Step 2: Oracle for target |11> (Phase flip via CZ)
qc.cz(0, 1)

# Step 3: Diffusion Operator
qc.h([0, 1])
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.h([0, 1])

# Step 4: Measurement
qc.measure([0, 1], [0, 1])

print("Grover's Search Circuit (Target = |11>):")
print(qc.draw('text'))

Grover's Search Circuit (Target = |11>):
     ┌───┐   ┌───┐┌───┐   ┌───┐┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├─■─┤ X ├┤ H ├┤M├───
     ├───┤ │ ├───┤├───┤ │ ├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├─■─┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘   └───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════╩══╩═
                                     0  1 


### 2. [Quantum Counting Algorithm](https://en.wikipedia.org/wiki/Quantum_counting_algorithm)
* **Problem Domain:** Solution Space Estimation.
* **Core Function:** Estimates the total number of marked items $M$ in an unstructured domain of size $N$.
* **Algorithmic Mechanism:**
    1. Combines Grover's Search iteration operator $G$ with Quantum Phase Estimation (QPE).
    2. Treats the Grover operator $G$ as the unitary for QPE.
    3. The resulting estimated phase $\theta$ maps to the number of solutions via $M = N \sin^2(\pi \theta)$.

In [2]:
from qiskit import QuantumCircuit

# Conceptual structure: Counting register (2 qubits) + Grover register (2 qubits)
qc = QuantumCircuit(4, 2)

# Initialize registers
qc.h([0, 1])  # Counting register superposition
qc.h([2, 3])  # Grover state initialization

# Controlled Grover Iterations (C-G^1, C-G^2)
# Applies Grover iteration conditioned on counting qubits
qc.cz(2, 3)   # Exemplar internal Grover iteration
qc.cx(0, 2)

# Inverse QFT on Counting Register
qc.h(0)
qc.cp(-1.57, 0, 1)
qc.h(1)

qc.measure([0, 1], [0, 1])

print("Quantum Counting Conceptual Circuit:")
print(qc.draw('text'))

Quantum Counting Conceptual Circuit:
     ┌───┐        ┌───┐                ┌─┐   
q_0: ┤ H ├─────■──┤ H ├─■──────────────┤M├───
     ├───┤     │  └───┘ │P(-1.57) ┌───┐└╥┘┌─┐
q_1: ┤ H ├─────┼────────■─────────┤ H ├─╫─┤M├
     ├───┤   ┌─┴─┐                └───┘ ║ └╥┘
q_2: ┤ H ├─■─┤ X ├──────────────────────╫──╫─
     ├───┤ │ └───┘                      ║  ║ 
q_3: ┤ H ├─■────────────────────────────╫──╫─
     └───┘                              ║  ║ 
c: 2/═══════════════════════════════════╩══╩═
                                        0  1 


### 3. [Amplitude Amplification](https://en.wikipedia.org/wiki/Amplitude_amplification)
* **Problem Domain:** Unstructured Search, Algorithm Probability Boosting.
* **Core Function:** Generalizes Grover's search to amplify the success probability of any arbitrary quantum algorithm $A$ from $\epsilon$ to $\sim 1$ in $\mathcal{O}(1/\sqrt{\epsilon})$ steps.
* **Algorithmic Mechanism:**
    1. Initialize state $\vert{}\Psi\rangle = A\vert{}0\rangle$.
    2. Apply target phase reflection operator $S_\chi = I - 2\vert{}\chi\rangle\langle\chi\vert{}$.
    3. Apply initial state reflection operator $S_0 = A(I - 2\vert{}0\rangle\langle 0\vert{})A^\dagger$.
    4. Iterate composite operator $Q = -A S_0 A^\dagger S_\chi$ to rotate state towards target space.

In [1]:
from qiskit import QuantumCircuit

# Amplitude Amplification using non-uniform initial preparation A
qc = QuantumCircuit(2, 2)

# Step 1: Arbitrary State Prep A (RY rotation creates non-uniform superposition)
qc.ry(1.0, 0)
qc.cx(0, 1)

# Step 2: Target Phase Reflection S_chi (Target state |11>)
qc.cz(0, 1)

# Step 3: Reflection S_0 about initial state (A S_zero A_dagger)
qc.cx(0, 1)
qc.ry(-1.0, 0)
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.ry(1.0, 0)
qc.cx(0, 1)

qc.measure([0, 1], [0, 1])

print("Amplitude Amplification Step Circuit:")
print(qc.draw('text'))

Amplitude Amplification Step Circuit:
     ┌───────┐             ┌────────┐┌───┐   ┌───┐┌───────┐     ┌─┐   
q_0: ┤ Ry(1) ├──■───■───■──┤ Ry(-1) ├┤ X ├─■─┤ X ├┤ Ry(1) ├──■──┤M├───
     └───────┘┌─┴─┐ │ ┌─┴─┐└─┬───┬──┘└───┘ │ ├───┤└───────┘┌─┴─┐└╥┘┌─┐
q_1: ─────────┤ X ├─■─┤ X ├──┤ X ├─────────■─┤ X ├─────────┤ X ├─╫─┤M├
              └───┘   └───┘  └───┘           └───┘         └───┘ ║ └╥┘
c: 2/════════════════════════════════════════════════════════════╩══╩═
                                                                 0  1 


### 4. [BHT Algorithm (Brassard–Heller–Tapp)](https://en.wikipedia.org/wiki/BHT_algorithm)
* **Problem Domain:** [Collision Problem](https://en.wikipedia.org/wiki/Collision_problem), Cryptanalysis [(Hash Functions)](https://en.wikipedia.org/wiki/Hash_function).
* **Core Function:** Finds a collision pair $(x, y)$ such that $f(x) = f(y)$ for 2-to-1 functions in $\mathcal{O}(N^{1/3})$ queries instead of classical $\mathcal{O}(N^{1/2})$.
* **Algorithmic Mechanism:**
    1. Randomly sample $K = N^{1/3}$ inputs classically and store $(x, f(x))$ in a classical lookup table/hash map.
    2. Construct an oracle $O$ that marks any input $y \notin K$ whose output matches an entry in the classical table ($f(y) \in f(K)$).
    3. Perform Grover search over the remaining domain size $N$ using $O$, taking $\mathcal{O}(\sqrt{N/K}) = \mathcal{O}(N^{1/3})$ quantum queries.

In [2]:
from qiskit import QuantumCircuit

# Structural representation of Grover search component within BHT algorithm
qc = QuantumCircuit(3, 2)

# Input search space superposition (N^{2/3} domain size)
qc.h([0, 1])

# Oracle marking items present in classical precomputed table K
qc.cx(0, 2)
qc.cz(1, 2)  # Marks collision matching precomputed hash set

# Diffusion Step
qc.h([0, 1])
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.h([0, 1])

qc.measure([0, 1], [0, 1])

print("BHT Quantum Search Stage Circuit:")
print(qc.draw('text'))

BHT Quantum Search Stage Circuit:
     ┌───┐     ┌───┐┌───┐        ┌───┐┌───┐┌─┐   
q_0: ┤ H ├──■──┤ H ├┤ X ├──────■─┤ X ├┤ H ├┤M├───
     ├───┤  │  └───┘├───┤┌───┐ │ ├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├──┼────■──┤ H ├┤ X ├─■─┤ X ├┤ H ├─╫─┤M├
     └───┘┌─┴─┐  │  └───┘└───┘   └───┘└───┘ ║ └╥┘
q_2: ─────┤ X ├──■──────────────────────────╫──╫─
          └───┘                             ║  ║ 
c: 2/═══════════════════════════════════════╩══╩═
                                            0  1 
